# Inferrail (per-run dollar budgets)

One query engine call or agent run can turn into many LLM calls: retrieval, sub-question decomposition, tool loops, retries, and parallel calls from `acomplete` / workflows. [Inferrail](https://github.com/domondi1/inferrail) is an open-source, OpenAI-compatible gateway that gives one run a hard dollar budget. Each request carries the run's id and budget as headers; the gateway reserves the request's worst-case cost before forwarding it and refuses it with HTTP 402 when the run's budget can't cover it, so concurrent calls in the same run can't overspend together. The budget headers are not forwarded to the provider.

Because it speaks the OpenAI API, you use it through `OpenAILike` with `default_headers`.

In [ ]:
%pip install llama-index-llms-openai-like inferrail

## Start the gateway

`inferrail.start()` runs the gateway on a background thread in this process and returns its base URL. It forwards to OpenAI with your key. For several worker processes, run one shared `inferrail serve` instead and use its URL.

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "sk-..."

import inferrail

base_url = inferrail.start()

## One LLM per run

Give the LLM the run's id and dollar budget. Every call made with it draws from that one budget. `max_tokens` bounds each call's reservation, and `max_retries=0` because a 402 is a budget decision, not a transient error.

In [ ]:
from llama_index.llms.openai_like import OpenAILike


def llm_for_run(run_id: str, budget_usd: str) -> OpenAILike:
    return OpenAILike(
        model="gpt-4o-mini",
        api_base=base_url,
        api_key="unused",
        is_chat_model=True,
        is_function_calling_model=True,
        context_window=128000,
        max_tokens=400,
        max_retries=0,
        default_headers={
            "X-Inferrail-Attribute-Work-Id": run_id,
            "X-Inferrail-Budget-Usd": budget_usd,
        },
    )


llm = llm_for_run("run-42", "0.05")
print(llm.complete("Summarize what a vector index is in one sentence."))

Use the same LLM for everything in the run, e.g. as the default for a query engine or agent:

In [ ]:
from llama_index.core import Settings

Settings.llm = llm_for_run("report-2026-10-05", "1.00")

## When the budget runs out

Concurrent calls in one run share the budget. Once it can't cover the next call, that call is refused before it reaches OpenAI and raises `openai.APIStatusError` with `status_code == 402`:

In [ ]:
import asyncio

from openai import APIStatusError

tight = llm_for_run("run-tight", "0.0007")


async def summarize(i: int) -> str:
    try:
        return str(await tight.acomplete(f"Summarize section {i} in one line."))
    except APIStatusError as e:
        if e.status_code == 402:
            return "refused: run budget exhausted"
        raise


results = await asyncio.gather(*[summarize(i) for i in range(8)])
results

## What the run cost

`inferrail work` shows how many calls a run made and what they cost, from receipts that store usage and cost but no prompt or response text:

In [ ]:
!inferrail work run-tight

Notes:

- Chat completions only. Keep your embedding model on its own endpoint.
- The model needs a price in Inferrail (`gpt-4o-mini`, `gpt-4.1-mini`, `gpt-4.1` are built in; `inferrail models` lists the rest, and you can add your own prices). Without one, a budgeted request is refused rather than guessed.
- Budgets live in a SQLite file on the gateway host, so use one gateway per set of runs.